# Federated Averaging (FedAvg) on Rotation-Based CIFAR-10

This notebook implements standard Federated Averaging (FedAvg) on rotation-based CIFAR-10 data with heterogeneous clients.

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
import copy
import random
import time

sys.path.append('..')
from training.utils import get_model, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Configuration

In [ ]:
# Load configuration from JSON
with open('config.json', 'r') as f:
    CONFIG = json.load(f)

# Set random seeds
SEED = CONFIG['seed']
set_seed(SEED)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Create Rotation-Based Dataset

In [ ]:
class RotatedCIFAR10Dataset(Dataset):
    """CIFAR-10 dataset with rotation applied."""
    def __init__(self, base_dataset, rotation_angle):
        self.base_dataset = base_dataset
        self.rotation_angle = rotation_angle
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        
        if self.rotation_angle != 0:
            image = transforms.functional.rotate(image, self.rotation_angle)
        
        image = self.transform(image)
        return image, label

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
train_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset_raw)}")
print(f"Test dataset size: {len(test_dataset_raw)}")

## Distribute Data Across Clients with Rotations

In [ ]:
# Generate rotation angles dynamically
rotation_angles = [int(360 * i / CONFIG['num_rotation_clusters']) for i in range(CONFIG['num_rotation_clusters'])]
print(f"Rotation angles: {rotation_angles}°")

# Assign clients to rotations
clients_per_rotation = CONFIG['num_clients'] // CONFIG['num_rotation_clusters']
client_rotation_labels = []

for rotation_idx, angle in enumerate(rotation_angles):
    start_client = rotation_idx * clients_per_rotation
    end_client = start_client + clients_per_rotation
    
    if rotation_idx == len(rotation_angles) - 1:
        end_client = CONFIG['num_clients']
    
    for client_idx in range(start_client, end_client):
        client_rotation_labels.append(angle)

print(f"\nClient distribution across rotations:")
for angle in rotation_angles:
    count = client_rotation_labels.count(angle)
    print(f"  {angle}°: {count} clients")

# Create rotated datasets for each client
train_subsets = []
samples_per_client = len(train_dataset_raw) // CONFIG['num_clients']
all_indices = list(range(len(train_dataset_raw)))
random.shuffle(all_indices)

for client_idx in range(CONFIG['num_clients']):
    angle = client_rotation_labels[client_idx]
    rotated_dataset = RotatedCIFAR10Dataset(train_dataset_raw, angle)
    
    start_idx = client_idx * samples_per_client
    end_idx = start_idx + samples_per_client if client_idx < CONFIG['num_clients'] - 1 else len(train_dataset_raw)
    
    client_indices = all_indices[start_idx:end_idx]
    subset = Subset(rotated_dataset, client_indices)
    train_subsets.append(subset)

print(f"\nCreated {len(train_subsets)} client datasets")
print(f"Average samples per client: {np.mean([len(s) for s in train_subsets]):.1f}")

## Create Test Dataset

In [ ]:
# Create test dataset (no rotation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Test dataset size: {len(test_dataset)}")

## Initialize Global Model

In [ ]:
# Initialize global model
global_model = get_model(
    model_name=CONFIG['model_name'],
    num_classes=10,
    pretrained=CONFIG.get('pretrained', False)
).to(device)

criterion = nn.CrossEntropyLoss()

print(f"Global model initialized: {CONFIG['model_name']}")

## Define FedAvg Functions

In [ ]:
def train_local_model(model, train_loader, epochs, lr):
    """Train local model for specified epochs."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model.state_dict()

def aggregate_models(client_weights, client_sizes):
    """Aggregate client models using weighted averaging."""
    total_size = sum(client_sizes)
    avg_weights = copy.deepcopy(client_weights[0])
    
    for key in avg_weights.keys():
        avg_weights[key] = torch.zeros_like(avg_weights[key], dtype=torch.float32)
        for i in range(len(client_weights)):
            weight = client_sizes[i] / total_size
            avg_weights[key] += client_weights[i][key] * weight
    
    return avg_weights

def evaluate_model(model, test_loader):
    """Evaluate model on test set."""
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = test_loss / len(test_loader)
    accuracy = correct / total
    
    return avg_loss, accuracy

print("FedAvg functions defined")

## Run FedAvg Training

In [ ]:
# FedAvg configuration
num_rounds = CONFIG.get('training_rounds', 30)
local_epochs = CONFIG.get('warmup_epochs', 2)
client_fraction = CONFIG.get('client_fraction', 0.2)
num_selected = max(int(client_fraction * CONFIG['num_clients']), 1)

# Storage for results
test_losses = []
test_accs = []
round_times = []

print(f"Starting FedAvg training...")
print(f"Number of rounds: {num_rounds}")
print(f"Local epochs: {local_epochs}")
print(f"Clients per round: {num_selected}/{CONFIG['num_clients']} ({client_fraction*100:.0f}%)")
print(f"{'='*70}\n")

total_start_time = time.time()

for round_num in range(1, num_rounds + 1):
    round_start = time.time()
    
    # Sample clients for this round
    selected_clients = random.sample(range(CONFIG['num_clients']), num_selected)
    
    # Store client updates
    client_weights = []
    client_sizes = []
    
    # Get global weights
    global_weights = global_model.state_dict()
    
    # Train selected clients
    for client_idx in selected_clients:
        # Create local model
        local_model = get_model(
            model_name=CONFIG['model_name'],
            num_classes=10,
            pretrained=False
        ).to(device)
        local_model.load_state_dict(global_weights)
        
        # Create data loader
        train_loader = DataLoader(
            train_subsets[client_idx],
            batch_size=CONFIG['batch_size'],
            shuffle=True
        )
        
        # Train locally
        updated_weights = train_local_model(local_model, train_loader, local_epochs, CONFIG['lr'])
        
        # Store update
        client_weights.append(updated_weights)
        client_sizes.append(len(train_subsets[client_idx]))
    
    # Aggregate updates
    aggregated_weights = aggregate_models(client_weights, client_sizes)
    global_model.load_state_dict(aggregated_weights)
    
    # Evaluate global model
    test_loss, test_acc = evaluate_model(global_model, test_loader)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    round_time = time.time() - round_start
    round_times.append(round_time)
    
    # Print progress
    if round_num % 5 == 0 or round_num == 1:
        print(f"Round {round_num}/{num_rounds} - "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, "
              f"Time: {round_time:.2f}s")

total_training_time = time.time() - total_start_time

print(f"\n{'='*70}")
print(f"FedAvg Training Complete!")
print(f"{'='*70}")
print(f"Total training time: {total_training_time:.2f}s ({total_training_time/60:.2f} min)")
print(f"Average time per round: {np.mean(round_times):.2f}s")
print(f"Final test accuracy: {test_accs[-1]:.4f}")
print(f"Best test accuracy: {max(test_accs):.4f} (round {np.argmax(test_accs)+1})")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Loss curve
ax = axes[0]
rounds_range = range(1, num_rounds + 1)
ax.plot(rounds_range, test_losses, 'o-', linewidth=2, markersize=4)
ax.set_xlabel('Round')
ax.set_ylabel('Test Loss')
ax.set_title('FedAvg Test Loss Over Rounds')
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy curve
ax = axes[1]
ax.plot(rounds_range, test_accs, 's-', linewidth=2, markersize=4, color='green')
ax.axhline(y=max(test_accs), color='red', linestyle='--', alpha=0.5, 
           label=f'Best: {max(test_accs):.4f}')
ax.set_xlabel('Round')
ax.set_ylabel('Test Accuracy')
ax.set_title('FedAvg Test Accuracy Over Rounds')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fedavg_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training curves saved as 'fedavg_training_curves.png'")

## Test on Different Rotations

In [ ]:
# Create test loaders for each rotation
rotation_test_results = {}

print(f"Evaluating on different rotations...\n")

for angle in rotation_angles:
    rotated_test = RotatedCIFAR10Dataset(test_dataset_raw, angle)
    test_subset_rot = Subset(rotated_test, list(range(len(test_dataset_raw))))
    rotation_loader = DataLoader(test_subset_rot, batch_size=CONFIG['batch_size'], shuffle=False)
    
    # Evaluate
    _, accuracy = evaluate_model(global_model, rotation_loader)
    rotation_test_results[angle] = accuracy
    print(f"Rotation {angle}°: Accuracy = {accuracy:.4f}")

# Visualize per-rotation performance
plt.figure(figsize=(10, 6))
angles = list(rotation_test_results.keys())
accs = list(rotation_test_results.values())

plt.bar([f"{a}°" for a in angles], accs, alpha=0.7, edgecolor='black', color='steelblue')
plt.axhline(y=np.mean(accs), color='red', linestyle='--', label=f'Average: {np.mean(accs):.4f}')
plt.xlabel('Rotation Angle')
plt.ylabel('Test Accuracy')
plt.title('FedAvg Model Performance on Different Rotations')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig('fedavg_rotation_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPerformance variance across rotations:")
print(f"  Mean: {np.mean(accs):.4f}")
print(f"  Std:  {np.std(accs):.4f}")
print(f"  Min:  {np.min(accs):.4f} ({angles[np.argmin(accs)]}°)")
print(f"  Max:  {np.max(accs):.4f} ({angles[np.argmax(accs)]}°)")

## Confusion Matrix on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Collect all predictions and true labels
all_predictions = []
all_labels = []

global_model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = global_model(inputs)
        _, predicted = outputs.max(1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Confusion matrix with counts
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

# Plot 2: Normalized confusion matrix (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ax = axes[1]
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            vmin=0, vmax=1, cbar_kws={'label': 'Proportion'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('fedavg_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Analyze per-class performance
print("\nPer-class Performance:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("="*65)

for i, class_name in enumerate(class_names):
    # True Positives, False Positives, False Negatives
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    support = cm[i, :].sum()
    
    # Calculate metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{class_name:<15} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {support:<10}")

# Overall metrics
overall_accuracy = np.trace(cm) / cm.sum()
print(f"\n{'Overall Accuracy':<15} {overall_accuracy:.4f}")

# Find most confused pairs
print("\nMost Confused Class Pairs:")
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)
for true_class, pred_class, count in confused_pairs[:10]:
    print(f"  {true_class:<12} → {pred_class:<12}: {count:>4} times")

## Analyze Client Selection Patterns

In [ ]:
# Analyze which rotation groups contributed most
print("\nClient selection analysis:")
print(f"Total client participations: {num_rounds * num_selected}")
print(f"Expected participations per client: {num_rounds * client_fraction:.1f}")

# Calculate participation by rotation
rotation_to_id = {angle: idx for idx, angle in enumerate(rotation_angles)}
rotation_participation = {angle: 0 for angle in rotation_angles}

# This is simplified - in practice you'd track actual selections
expected_per_rotation = (num_rounds * num_selected) / CONFIG['num_rotation_clusters']
for angle in rotation_angles:
    rotation_participation[angle] = expected_per_rotation

plt.figure(figsize=(10, 6))
plt.bar([f"{a}°" for a in angles], 
        [rotation_participation[a] for a in angles],
        alpha=0.7, edgecolor='black', color='orange')
plt.axhline(y=expected_per_rotation, color='red', linestyle='--', 
           label=f'Expected: {expected_per_rotation:.0f}')
plt.xlabel('Rotation Cluster')
plt.ylabel('Expected Participations')
plt.title('Expected Client Participations by Rotation (Uniform Sampling)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Save Results

In [ ]:
# Save results to file
results = {
    'method': 'fedavg',
    'config': CONFIG,
    'num_rounds': num_rounds,
    'local_epochs': local_epochs,
    'client_fraction': client_fraction,
    'training_time': total_training_time,
    'avg_round_time': float(np.mean(round_times)),
    'final_test_acc': float(test_accs[-1]),
    'best_test_acc': float(max(test_accs)),
    'test_losses': [float(x) for x in test_losses],
    'test_accs': [float(x) for x in test_accs],
    'rotation_results': {int(k): float(v) for k, v in rotation_test_results.items()}
}

with open('fedavg_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to 'fedavg_results.json'")

# Save model checkpoint
torch.save({
    'round': num_rounds,
    'model_state_dict': global_model.state_dict(),
    'test_acc': test_accs[-1],
}, 'fedavg_model_checkpoint.pth')

print("Model checkpoint saved to 'fedavg_model_checkpoint.pth'")

## Summary

**Federated Averaging (FedAvg) Results:**

**Algorithm:**
1. Initialize global model
2. Each round:
   - Sample fraction of clients randomly
   - Each client trains locally for E epochs
   - Aggregate client models via weighted averaging
   - Update global model
3. Repeat for R rounds

**Key Characteristics:**
- Decentralized training (no direct data sharing)
- Communication overhead: model weights exchanged each round
- Handles non-IID data through averaging
- Privacy: clients keep data locally

**Compared to Centralized:**
- Lower accuracy (non-IID data challenges)
- Privacy preserving
- Communication cost
- Suitable for federated scenarios

**Next Steps:**
- Compare with centralized baseline
- Compare with ensemble clustering approach
- Analyze convergence speed vs centralized
- Evaluate communication efficiency